# 5.1 SAP – Databricks Connection Patterns

Reference notebook outlining three common patterns for connecting SAP source systems (BW, HANA, S/4HANA) to Databricks for downstream analytics.

Patterns covered:
1. **Spark JDBC** — batch pull from HANA / SAP SQL views
2. **Delta Live Tables (DLT)** — declarative ingestion pipelines
3. **Delta Sharing** — governed open sharing of curated tables

Cells run locally with `pandas`/`sqlite3`. PySpark/Databricks equivalents are annotated in comments so the same logic can be ported to a Databricks workspace without rewriting.

## Pattern 1: Spark JDBC — batch pull from SAP HANA

Use when: you need scheduled, snapshot extracts of SAP tables/CDS views into the Bronze layer.

In [ ]:
import pandas as pd
import sqlite3

# Local stand-in for SAP HANA: read materials master via sqlite3
# In Databricks this would be:
#   df = (spark.read.format('jdbc')
#       .option('url', 'jdbc:sap://<host>:<port>')
#       .option('driver', 'com.sap.db.jdbc.Driver')
#       .option('dbtable', 'SAPABAP1.MARA')
#       .option('user', dbutils.secrets.get('sap', 'user'))
#       .option('password', dbutils.secrets.get('sap', 'pwd'))
#       .load())

con = sqlite3.connect(':memory:')
pd.DataFrame({
    'MATNR': ['MAT-00001', 'MAT-00002', 'MAT-00003'],
    'MAKTX': ['Bearing 25mm', 'Pump housing', 'Hydraulic seal'],
    'WERKS': ['1000', '1000', '2000']
}).to_sql('mara', con, index=False)

df = pd.read_sql('SELECT * FROM mara', con)
df

### Partitioning hint
When the source table is large, pass `partitionColumn`, `lowerBound`, `upperBound`, `numPartitions` to parallelize the JDBC read across executors.

In [ ]:
# PySpark equivalent (run in Databricks):
# df = (spark.read.format('jdbc')
#       .option('url', jdbc_url)
#       .option('dbtable', 'SAPABAP1.VBAP')
#       .option('partitionColumn', 'VBELN')
#       .option('lowerBound', 1000000)
#       .option('upperBound', 9999999)
#       .option('numPartitions', 16)
#       .load())
print('See PySpark snippet above — run in a Databricks workspace.')

## Pattern 2: Delta Live Tables (DLT) — declarative ingestion

Use when: you want a managed pipeline with built-in expectations, lineage, and incremental processing.

DLT pipelines are defined declaratively. Below is the pattern — these decorators run inside a Databricks DLT pipeline.

In [ ]:
# Reference — paste into a Databricks DLT pipeline notebook:
#
# import dlt
# from pyspark.sql.functions import col
#
# @dlt.table(comment='Raw SAP material master from HANA')
# def bronze_mara():
#     return (spark.read.format('jdbc')
#             .option('url', jdbc_url)
#             .option('dbtable', 'SAPABAP1.MARA')
#             .load())
#
# @dlt.table(comment='Cleaned material master')
# @dlt.expect_or_drop('matnr_not_null', 'MATNR IS NOT NULL')
# def silver_materials():
#     return (dlt.read('bronze_mara')
#             .select('MATNR', 'MAKTX', 'WERKS', 'MTART')
#             .dropDuplicates(['MATNR']))
print('DLT pattern shown above — deploy as a pipeline in Databricks.')

## Pattern 3: Delta Sharing — governed open sharing

Use when: you need to share curated SAP-derived tables (e.g., BW KPI marts) with downstream consumers (other teams, external partners) without copying data.

In [ ]:
# Provider side (Databricks SQL):
#   CREATE SHARE sap_kpis;
#   ALTER SHARE sap_kpis ADD TABLE finance.gold.bw_sales_kpis;
#   CREATE RECIPIENT downstream_team USING IDENTIFIER 'team@example.com';
#   GRANT SELECT ON SHARE sap_kpis TO RECIPIENT downstream_team;
#
# Recipient side (any Delta Sharing client):
#   import delta_sharing
#   profile = '/path/to/config.share'
#   df = delta_sharing.load_as_pandas(f'{profile}#sap_kpis.gold.bw_sales_kpis')
print('Delta Sharing flow above — provider grants, recipient consumes.')

## When to pick which pattern

| Pattern | Best for | Trade-off |
|---------|----------|-----------|
| Spark JDBC | One-off / scheduled batch | You manage scheduling and incremental logic |
| DLT | Production pipelines with quality rules | Locked into Databricks runtime |
| Delta Sharing | Cross-team / cross-org consumption | Read-only; provider controls schema |